# Merging metadata and filtering out original images if re-imaged data exist

## Overview
In this notebook, we merge the metadata information extracted from the previous notebook and identify duplicate items that arise from re-imaging, then drop entries corresponding to the original images provided re-imaging entries exist to ensure the low qualitied images are not included downstream. 

## Import libraries

In [1]:
import os
import pathlib
import pandas as pd

## Define data and output location

In [2]:
# Location tothe extracted metadata information
METADATA_PATH = os.path.join(
    pathlib.Path(".").absolute(),
    "metadata"
)
PLATEMAP_PATH = os.path.join(METADATA_PATH, 'platemaps')

In [3]:
## load image index (PlateID, row, column, field, channel, file name and other image specific metadata)
df_index = pd.read_csv(os.path.join(METADATA_PATH, 'image_index.csv'))

# load re image status (PlateID)
df_re_image = pd.read_csv(os.path.join(METADATA_PATH, 're_image_status.csv'))

# load barcode to plate mapping and plate to well, cell line and seeding density mapping
barcode_plate_mapping = pd.read_csv(os.path.join(
    PLATEMAP_PATH,
    'Barcode_platemap_pilot_data.csv')
)

plate_well_mapping = pd.DataFrame()
for fname in barcode_plate_mapping['platemap_file'].unique():
    plate_well_mapping = pd.concat(
        [plate_well_mapping, 
         pd.read_csv(os.path.join(PLATEMAP_PATH, f'{fname}.csv')).assign(platemap_file=fname)],
        ignore_index=True
    )

## Merge platemap data

In [4]:
df_platemap = pd.merge(
    left=barcode_plate_mapping,
    right=plate_well_mapping,
    how='inner',
    on='platemap_file'
).rename(
    columns={
        'barcode': 'PlateID',
        'row': 'Row',
        'column': 'Col'
    }
)
df_platemap.head()

,PlateID,time_point,platemap_file,cell_line,Row,Col,well,seeding_density
0,BR00143976,24,Assay_Plate1_platemap,DAOY,C,3,C03,1000
1,BR00143976,24,Assay_Plate1_platemap,DAOY,D,3,D03,1000
2,BR00143976,24,Assay_Plate1_platemap,IMR32,E,3,E03,1000
3,BR00143976,24,Assay_Plate1_platemap,IMR32,F,3,F03,1000
4,BR00143976,24,Assay_Plate1_platemap,PA1,G,3,G03,1000


## Merge image index metadata, re-image metadata with platemap metadata

In [5]:
df_merge = pd.merge(
    left=df_index,
    right=df_re_image,
    how='inner',
    on=['PlateID', 'URL', 'Measurement']
)

row_number_to_letter_mapping = {i: chr(64 + i) for i in range(1, 27)}
df_merge['Row'] = df_merge['Row'].map(row_number_to_letter_mapping)

df_merge = pd.merge(
    left=df_merge,
    right=df_platemap,
    how='inner',
    on=['PlateID', 'Row', 'Col']
)
df_merge.head()

,PlateID,MeasurementID,MeasurementStartTime,Version,id,State,URL,Row,Col,FieldID,...,AbsTime,Measurement,re_imaged,Date,tiff_path,time_point,platemap_file,cell_line,well,seeding_density
0,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R1,Ok,r03c14f01p01-ch1sk1fk1fl1.tiff,C,14,1,...,2024-07-17T18:50:48.9006008-04:00,2,True,2024-07-17,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,A673,C14,1000
1,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R2,Ok,r03c14f01p01-ch2sk1fk1fl1.tiff,C,14,1,...,2024-07-17T18:50:49.1970008-04:00,2,True,2024-07-17,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,A673,C14,1000
2,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R3,Ok,r03c14f01p01-ch3sk1fk1fl1.tiff,C,14,1,...,2024-07-17T18:50:49.3218008-04:00,2,True,2024-07-17,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,A673,C14,1000
3,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R4,Ok,r03c14f01p01-ch4sk1fk1fl1.tiff,C,14,1,...,2024-07-17T18:50:49.4466008-04:00,2,True,2024-07-17,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,A673,C14,1000
4,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R5,Ok,r03c14f01p01-ch5sk1fk1fl1.tiff,C,14,1,...,2024-07-17T18:50:49.7586008-04:00,2,True,2024-07-17,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,A673,C14,1000


## Retain only entries corresponding to re-imaged data if re-image is done for the same combination of well, channel, field, and PlateID

In [6]:
df_merge.sort_values(
    by=['re_imaged'],
    ascending=False,
    inplace=True
)
print(f'# Images before dropping: \t{df_merge.shape[0]}')
df_deduplicated = df_merge.drop_duplicates(
    subset = ['PlateID', 'well', 'FieldID', 'ChannelID'],
    keep='first'
)
print(f'# Images after dropping: \t{df_deduplicated.shape[0]}')
df_deduplicated.head()


# Images before dropping: 	86514
# Images after dropping: 	61488


,PlateID,MeasurementID,MeasurementStartTime,Version,id,State,URL,Row,Col,FieldID,...,AbsTime,Measurement,re_imaged,Date,tiff_path,time_point,platemap_file,cell_line,well,seeding_density
0,BR00143978,57ddb76f-2d5b-4904-8743-bb6bafc8a296,2024-07-17T18:48:48.3580223-04:00,1,0314K1F1P1R1,Ok,r03c14f01p01-ch1sk1fk1fl1.tiff,C,14,1,...,2024-07-17T18:50:48.9006008-04:00,2,True,2024-07-17,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,A673,C14,1000
71620,BR00143978,86951e3b-d00d-44f3-bb0e-a60f76834151,2024-07-17T20:02:24.8472787-04:00,1,1106K1F8P1R5,Ok,r11c06f08p01-ch5sk1fk1fl1.tiff,K,6,8,...,2024-07-17T20:04:57.806801-04:00,4,True,2024-07-17,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,CHP212,K06,2000
71607,BR00143978,86951e3b-d00d-44f3-bb0e-a60f76834151,2024-07-17T20:02:24.8472787-04:00,1,1106K1F6P1R4,Ok,r11c06f06p01-ch4sk1fk1fl1.tiff,K,6,6,...,2024-07-17T20:04:53.766401-04:00,4,True,2024-07-17,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,CHP212,K06,2000
71608,BR00143978,86951e3b-d00d-44f3-bb0e-a60f76834151,2024-07-17T20:02:24.8472787-04:00,1,1106K1F6P1R5,Ok,r11c06f06p01-ch5sk1fk1fl1.tiff,K,6,6,...,2024-07-17T20:04:54.109601-04:00,4,True,2024-07-17,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,CHP212,K06,2000
71609,BR00143978,86951e3b-d00d-44f3-bb0e-a60f76834151,2024-07-17T20:02:24.8472787-04:00,1,1106K1F6P1R6,Ok,r11c06f06p01-ch6sk1fk1fl1.tiff,K,6,6,...,2024-07-17T20:04:54.468401-04:00,4,True,2024-07-17,/home/newusr/Waylab/ALSF_img2img/data/ALSF_pil...,48,Assay_Plate1_platemap,CHP212,K06,2000


In [7]:
df_deduplicated.groupby(
    ['cell_line', 'seeding_density', 'time_point', 're_imaged', 'Measurement']
).count()

PlateID  \
cell_line seeding_density time_point re_imaged Measurement            
A673      1000            24         True      3                216   
                          48         False     1                156   
                                     True      2                 60   
                          72         True      2                216   
          2000            24         True      3                216   
...                                                             ...   
U2-OS     8000            72         False     1                432   
          12000           24         False     1                216   
                                               2                216   
                          48         False     1                432   
                          72         False     1                432   

                                                            MeasurementID  \
cell_line seeding_density time_point re_imaged Measurement                  
A673      1000            24         True      3                      216   
                          48         False     1                      156   
                                     True      2                       60   
                          72         True      2                      216   
          2000            24         True      3                      216   
...                                                                   ...   
U2-OS     8000            72         False     1                      432   
          12000           24         False     1                      216   
                                               2                      216   
                          48         False     1                      432   
                          72         False     1                      432   

                                                            MeasurementStartTime  \
cell_line seeding_density time_point re_imaged Measurement                         
A673      1000            24         True      3                             216   
                          48         False     1                             156   
                                     True      2                              60   
                          72         True      2                             216   
          2000            24         True      3                             216   
...                                                                          ...   
U2-OS     8000            72         False     1                             432   
          12000           24         False     1                             216   
                                               2                             216   
                          48         False     1                             432   
                          72         False     1                             432   

                                                            Version   id  \
cell_line seeding_density time_point re_imaged Measurement                 
A673      1000            24         True      3                216  216   
                          48         False     1                156  156   
                                     True      2                 60   60   
                          72         True      2                216  216   
          2000            24         True      3                216  216   
...                                                             ...  ...   
U2-OS     8000            72         False     1                432  432   
          12000           24         False     1                216  216   
                                               2                216  216   
                          48         False     1                432  432   
                          72         False     1                432  432   

                                                    

## Export filtered merged metadata

In [8]:
df_deduplicated.to_csv(
    os.path.join(METADATA_PATH, 'filtered_metadata.csv')
)